# Phase 2: Data Cleaning & Preprocessing Pipeline
**Project:** AI-Ecommerce-Analytics-Dashboard  
**Dataset:** Amazon Sales Report  
**Objective:** Transform raw transactional data into a standardized, high-quality, analytics-ready dataset while preserving 100% of legitimate records and business context.

## 1. Environment Setup & Data Loading

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Visual configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)

# Load raw dataset
raw_path = os.path.join('..', 'Data', 'Raw Data', 'amazon_sales.csv.csv')
df_raw = pd.read_csv(raw_path, low_memory=False)
print(f"Raw dataset shape: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns")
df = df_raw.copy()

## 2. Column Standardization & Removing Redundant Artifacts
- Strip leading/trailing whitespace from column headers (e.g. `'Sales Channel '` $\rightarrow$ `'Sales Channel'`).
- Remove redundant serial index (`index`) and uninformative export column (`Unnamed: 22`).
- Standardize column names to clean PascalCase format.

In [ ]:
# Strip whitespace
df.columns = [c.strip() for c in df.columns]

# Drop redundant columns
drop_cols = ['index', 'Unnamed: 22']
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

# Rename columns
renaming = {
    'Order ID': 'Order_ID',
    'Sales Channel': 'Sales_Channel',
    'ship-service-level': 'Ship_Service_Level',
    'Courier Status': 'Courier_Status',
    'ship-city': 'Ship_City',
    'ship-state': 'Ship_State',
    'ship-postal-code': 'Ship_Postal_Code',
    'ship-country': 'Ship_Country',
    'promotion-ids': 'Promotion_IDs',
    'fulfilled-by': 'Fulfilled_By',
    'Amount': 'Recorded_Amount',
    'currency': 'Currency'
}
df.rename(columns=renaming, inplace=True)
print(f"Columns after standardization ({len(df.columns)} cols):\n{list(df.columns)}")

## 3. Date Parsing & Temporal Feature Engineering
- Convert `Date` string (`MM-DD-YY`) to `datetime64[ns]`.
- Derive `Year`, `Month`, `Month_Name`, `Week`, `Day`, and `Day_Name`.

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], format='%m-%d-%y', errors='coerce')

# Temporal derivations
df['Year'] = df['Date'].dt.year.astype('int32')
df['Month'] = df['Date'].dt.month.astype('int32')
df['Month_Name'] = df['Date'].dt.month_name()
df['Week'] = df['Date'].dt.isocalendar().week.astype('int32')
df['Day'] = df['Date'].dt.day.astype('int32')
df['Day_Name'] = df['Date'].dt.day_name()

print(f"Min Date: {df['Date'].min().strftime('%Y-%m-%d')} | Max Date: {df['Date'].max().strftime('%Y-%m-%d')}")
print(f"Derived Temporal Columns Sample:")
df[['Date', 'Year', 'Month_Name', 'Week', 'Day_Name']].head(3)

## 4. Text & Categorical Standardization
- Standardize text casing (e.g. `kurta` $\rightarrow$ `Kurta`).
- Trim whitespace across all categorical attributes.

In [ ]:
df['Category'] = df['Category'].astype(str).str.strip().str.title()
df['Size'] = df['Size'].astype(str).str.strip().str.upper()
df['Status'] = df['Status'].astype(str).str.strip()
df['Fulfilment'] = df['Fulfilment'].astype(str).str.strip()
df['Sales_Channel'] = df['Sales_Channel'].astype(str).str.strip()
df['Ship_Service_Level'] = df['Ship_Service_Level'].astype(str).str.strip()
df['Ship_City'] = df['Ship_City'].fillna('Unknown/Not Provided').astype(str).str.strip().str.title()
df['Ship_Country'] = df['Ship_Country'].fillna('Unknown/Not Provided').astype(str).str.strip().str.upper()

print("Cleaned Category Values:", df['Category'].unique())
print("Cleaned Size Values:", df['Size'].unique())

## 5. Indian State Name Standardization
Map 69 raw state variations (abbreviations like `NL`, `AR`, `RJ`, `PB`, case variations, typos, and trailing whitespace) into standardized State/UT names.

In [ ]:
STATE_MAPPING = {
    'MAHARASHTRA': 'Maharashtra',
    'KARNATAKA': 'Karnataka',
    'TAMIL NADU': 'Tamil Nadu',
    'TELANGANA': 'Telangana',
    'UTTAR PRADESH': 'Uttar Pradesh',
    'DELHI': 'Delhi',
    'Delhi': 'Delhi',
    'delhi': 'Delhi',
    'New Delhi': 'Delhi',
    'KERALA': 'Kerala',
    'WEST BENGAL': 'West Bengal',
    'ANDHRA PRADESH': 'Andhra Pradesh',
    'Gujarat': 'Gujarat',
    'HARYANA': 'Haryana',
    'RAJASTHAN': 'Rajasthan',
    'Rajasthan': 'Rajasthan',
    'rajasthan': 'Rajasthan',
    'Rajshthan': 'Rajasthan',
    'rajsthan': 'Rajasthan',
    'Rajsthan': 'Rajasthan',
    'RJ': 'Rajasthan',
    'MADHYA PRADESH': 'Madhya Pradesh',
    'ODISHA': 'Odisha',
    'Odisha': 'Odisha',
    'Orissa': 'Odisha',
    'orissa': 'Odisha',
    'BIHAR': 'Bihar',
    'Bihar': 'Bihar',
    'bihar': 'Bihar',
    'PUNJAB': 'Punjab',
    'Punjab': 'Punjab',
    'punjab': 'Punjab',
    'Punjab/Mohali/Zirakpur': 'Punjab',
    'PB': 'Punjab',
    'ASSAM': 'Assam',
    'UTTARAKHAND': 'Uttarakhand',
    'JHARKHAND': 'Jharkhand',
    'GOA': 'Goa',
    'Goa': 'Goa',
    'goa': 'Goa',
    'CHHATTISGARH': 'Chhattisgarh',
    'HIMACHAL PRADESH': 'Himachal Pradesh',
    'JAMMU & KASHMIR': 'Jammu and Kashmir',
    'PUDUCHERRY': 'Puducherry',
    'Puducherry': 'Puducherry',
    'Pondicherry': 'Puducherry',
    'CHANDIGARH': 'Chandigarh',
    'Chandigarh': 'Chandigarh',
    'MANIPUR': 'Manipur',
    'Manipur': 'Manipur',
    'ANDAMAN & NICOBAR ': 'Andaman and Nicobar Islands',
    'MEGHALAYA': 'Meghalaya',
    'Meghalaya': 'Meghalaya',
    'SIKKIM': 'Sikkim',
    'Sikkim': 'Sikkim',
    'NAGALAND': 'Nagaland',
    'Nagaland': 'Nagaland',
    'NL': 'Nagaland',
    'TRIPURA': 'Tripura',
    'ARUNACHAL PRADESH': 'Arunachal Pradesh',
    'Arunachal Pradesh': 'Arunachal Pradesh',
    'Arunachal pradesh': 'Arunachal Pradesh',
    'AR': 'Arunachal Pradesh',
    'MIZORAM': 'Mizoram',
    'Mizoram': 'Mizoram',
    'DADRA AND NAGAR': 'Dadra and Nagar Haveli and Daman and Diu',
    'LADAKH': 'Ladakh',
    'LAKSHADWEEP': 'Lakshadweep',
    'APO': 'Unknown/Other'
}

df['Ship_State'] = df['Ship_State'].map(STATE_MAPPING).fillna('Unknown/Not Provided')
print(f"Total unique standardized states: {df['Ship_State'].nunique()}")
df['Ship_State'].value_counts().head(10)

## 6. PIN Code Standardization & Validation
Format postal codes into 6-digit zero-padded text strings and validate against Indian postal PIN format (`^[1-9][0-9]{5}$`).

In [ ]:
def format_pin(val):
    if pd.isnull(val):
        return 'UNKNOWN'
    try:
        val_int = int(float(val))
        return str(val_int).zfill(6)
    except:
        return str(val).strip()

df['Ship_Postal_Code'] = df['Ship_Postal_Code'].apply(format_pin)
pin_regex = re.compile(r'^[1-9][0-9]{5}$')
df['Is_Valid_PIN'] = df['Ship_Postal_Code'].apply(lambda p: bool(pin_regex.match(p)))

print(f"Valid PIN codes: {df['Is_Valid_PIN'].sum():,} ({df['Is_Valid_PIN'].mean()*100:.2f}%)")
print(f"Missing / Unknown PIN codes: {(~df['Is_Valid_PIN']).sum():,}")

## 7. Missing Value Imputation & Revenue Logic
- `Fulfilled_By`: Structurally missing for Amazon FBA $\rightarrow$ `'Not Applicable'`.
- `Promotion_IDs`: Missing $\rightarrow$ `'No Promotion'`.
- `Courier_Status`: Missing $\rightarrow$ `'Unassigned'`.
- `Currency`: Missing $\rightarrow$ `'INR'`.
- `Gross_Amount`: Original recorded amount with nulls treated as `0.0`.
- `Realized_Revenue`: Revenue realized from non-cancelled, non-returned transactions with `Qty > 0` and `Gross_Amount > 0`.

In [ ]:
# Impute categorical missing values
df['Fulfilled_By'] = df['Fulfilled_By'].fillna('Not Applicable')
df['Promotion_IDs'] = df['Promotion_IDs'].fillna('No Promotion')
df['Courier_Status'] = df['Courier_Status'].fillna('Unassigned')
df['Currency'] = df['Currency'].fillna('INR')
df['B2B'] = df['B2B'].astype(bool)

# Financial metrics
df['Gross_Amount'] = df['Recorded_Amount'].fillna(0.0)

# Flags
df['Is_Cancelled'] = df['Status'] == 'Cancelled'
df['Is_Delivered'] = df['Status'] == 'Shipped - Delivered to Buyer'
df['Is_Returned'] = df['Status'].isin([
    'Shipped - Returned to Seller', 
    'Shipped - Returning to Seller', 
    'Shipped - Rejected by Buyer'
])
df['Is_B2B'] = df['B2B']
df['Has_Promotion'] = df['Promotion_IDs'] != 'No Promotion'

# Realized Revenue Rule
non_realized_statuses = [
    'Cancelled',
    'Shipped - Returned to Seller',
    'Shipped - Returning to Seller',
    'Shipped - Rejected by Buyer',
    'Shipped - Lost in Transit',
    'Shipped - Damaged',
    'Shipping'
]

is_realized_condition = (
    (~df['Status'].isin(non_realized_statuses)) & 
    (df['Qty'] > 0) & 
    (df['Gross_Amount'] > 0)
)

df['Realized_Revenue'] = np.where(is_realized_condition, df['Gross_Amount'], 0.0)
df['Is_Realized'] = is_realized_condition

print(f"Gross Recorded Amount: INR {df['Gross_Amount'].sum():,.2f}")
print(f"Realized Net Revenue:  INR {df['Realized_Revenue'].sum():,.2f}")
print(f"Cancelled / Unrealized: INR {(df['Gross_Amount'].sum() - df['Realized_Revenue'].sum()):,.2f}")

## 8. Validation Checks & Final Quality Audit
Run post-cleaning verification checks.

In [ ]:
print("=== VALIDATION CHECKS ===")
print(f"1. Total rows preserved: {len(df):,} (Expected: {len(df_raw):,})")
assert len(df) == len(df_raw), "Data loss detected!"

print(f"2. Null values in key dimensions:")
key_cols = ['Order_ID', 'Date', 'Status', 'Category', 'Size', 'Gross_Amount', 'Realized_Revenue', 'Ship_State']
print(df[key_cols].isnull().sum())

print(f"3. Datetime valid: {pd.api.types.is_datetime64_any_dtype(df['Date'])}")
print(f"4. Postal code is string: {pd.api.types.is_string_dtype(df['Ship_Postal_Code']) or pd.api.types.is_object_dtype(df['Ship_Postal_Code'])}")
print(f"5. Realized revenue <= Gross amount: {(df['Realized_Revenue'] <= df['Gross_Amount']).all()}")
print("\nAll validation checks PASSED!")